In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
from datetime import datetime

warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
date_stamp = datetime.now().strftime("%Y_%m_%d")
FILE_AUTO_MON_PULL = "auto_monetization.csv" 
FILE_TRACKER = "tracker.csv"
FILE_DELTA_SUSPENSE = "delta_suspense.xlsx"
FILE_ETRANS_SUSPENSE = "etrans_suspense.xlsx"

# --- 1. CONFIGURATION ---
date_stamp = datetime.now().strftime("%Y_%m_%d")
FILE_AUTO_MON_PULL = "auto_monetization.csv" 
FILE_TRACKER = "tracker.csv"
FILE_DELTA_SUSPENSE = "delta_suspense.xlsx"
FILE_ETRANS_SUSPENSE = "etrans_suspense.xlsx"

# Output paths
OUTPUT_DIR = r"AutoMonReports"
SUMMARY_EXCEL = f"AutoMonMasters/AutoMon_Master_{date_stamp}.xlsx"

MAX_ROWS = 1048575
# -----------------

In [33]:
# --- 2. LOAD & INITIAL FILTER ---
raw_data = pd.read_csv(FILE_AUTO_MON_PULL)
tracker = pd.read_csv(FILE_TRACKER)
tracker.rename(columns={'Video ID': 'VIDEO_ID'}, inplace=True)

videos_matching = pd.merge(raw_data, tracker, how='inner', on='VIDEO_ID')
filtered_data = raw_data[~raw_data['VIDEO_ID'].isin(videos_matching['VIDEO_ID'])]
filtered_data['VIDEO_ID_LOWER'] = filtered_data['VIDEO_ID'].str.lower()

In [34]:
# --- 3. AGGREGATED SUSPENSE PREPARATION ---
# Summing the money first
eTrans_data = pd.read_excel(FILE_ETRANS_SUSPENSE)
eTrans_data['VIDEO_ID_LOWER'] = eTrans_data['DSP_PRODUCT_ID'].str.lower()
final_eT = eTrans_data.groupby('VIDEO_ID_LOWER', as_index=False)['AMOUNT'].sum()
final_eT.rename(columns={'AMOUNT': 'US Suspense'}, inplace=True)

delta_data = pd.read_excel(FILE_DELTA_SUSPENSE)
delta_data['VIDEO_ID_LOWER'] = delta_data['PRODUCT_ID'].str.lower()
final_delta = delta_data.groupby('VIDEO_ID_LOWER', as_index=False)['USD_VALUE'].sum()
final_delta.rename(columns={'USD_VALUE': 'Int Suspense'}, inplace=True)

In [35]:
# --- 4. MERGING REPORTS (Fixed to prevent exact row duplication) ---
# We use 'left' merge and ensure we only have unique keys on the right side
et_vr = pd.merge(filtered_data, final_eT, how='left', on='VIDEO_ID_LOWER')
et_delta_vr = pd.merge(et_vr, final_delta, how='left', on='VIDEO_ID_LOWER')

# Drop any technical artifacts
et_delta_vr.dropna(subset=['VIDEO_ID'], inplace=True)

# CRITICAL: Remove duplicates here to ensure clean data for summary and label splits
final_dataset = et_delta_vr.drop_duplicates().copy()

# REMOVE REQUESTED COLUMNS
# We remove them here so they are gone from all subsequent files
cols_to_remove = ['CLAIM_ID', 'VIDEO_ID_LOWER']
final_dataset.drop(columns=cols_to_remove, inplace=True, errors='ignore')

In [36]:
# --- 5. GENERATE STATISTICAL SUMMARY ---
summary_data = final_dataset.groupby('REPORT_NAME').agg(
    Videos_Without_Suspense=('US Suspense', lambda x: x.isna().sum()),
    Videos_With_Suspense=('US Suspense', lambda x: x.notna().sum()),
    Total_Videos=('VIDEO_ID', 'count'),
    Total_US_Suspense_Amount=('US Suspense', 'sum'),
    Total_Int_Suspense_Amount=('Int Suspense', 'sum')
).reset_index()

summary_data['Total_Combined_Suspense'] = summary_data['Total_US_Suspense_Amount'] + summary_data['Total_Int_Suspense_Amount']
summary_data[['Total_US_Suspense_Amount', 'Total_Int_Suspense_Amount', 'Total_Combined_Suspense']] = \
    summary_data[['Total_US_Suspense_Amount', 'Total_Int_Suspense_Amount', 'Total_Combined_Suspense']].round(2)

os.makedirs(os.path.dirname(SUMMARY_EXCEL), exist_ok=True)
summary_data.to_excel(SUMMARY_EXCEL, index=False)

In [37]:
# --- 6. LABEL REPORTS (With Chunking Fail-safe) ---
ada_only = final_dataset[final_dataset['REPORT_NAME'].str.contains('^ADA', na=False)]
non_ada_names = [name for name in final_dataset['REPORT_NAME'].unique() if not str(name).startswith('ADA')]

report_dfs = {name: final_dataset[final_dataset['REPORT_NAME'] == name] for name in non_ada_names}
report_dfs['ADA'] = ada_only

os.makedirs(OUTPUT_DIR, exist_ok=True)

for report_name, report_df in report_dfs.items():
    valid_name = "".join(c for c in str(report_name) if c.isalnum() or c in (' ', '_')).strip()
    file_path = os.path.join(OUTPUT_DIR, f"{valid_name}_Auto_Monetization_report_{date_stamp}.xlsx")
    
    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        # Categorize the data first
        suspense_2026 = report_df[report_df[['US Suspense', 'Int Suspense']].notna().any(axis=1)]
        no_suspense_2026 = report_df[report_df[['US Suspense', 'Int Suspense']].isna().all(axis=1) & 
                                     (pd.to_datetime(report_df['UPLOAD_DATE']).dt.year == 2026)]
        no_suspense_old = report_df[report_df[['US Suspense', 'Int Suspense']].isna().all(axis=1) & 
                                    (pd.to_datetime(report_df['UPLOAD_DATE']).dt.year < 2026)]

        categories = [('2026 Suspense', suspense_2026), ('2026 No Suspense', no_suspense_2026), ('Old No Suspense', no_suspense_old)]
        
        for base_name, df in categories:
            if not df.empty:
                # If the category is within Excel limits, write as a single tab
                if len(df) <= MAX_ROWS:
                    df.to_excel(writer, sheet_name=base_name, index=False, freeze_panes=(1,0))
                    num_format = writer.book.add_format({'num_format': '0.00'})
                    writer.sheets[base_name].set_column('K:L', 15, num_format)
                else:
                    # If category is too large, split into parts (e.g., India No Suspense P1, P2)
                    for i in range(0, len(df), MAX_ROWS):
                        chunk = df.iloc[i : i + MAX_ROWS]
                        # Shorten sheet name if needed (Excel allows max 31 characters)
                        part_name = f"{base_name[:28]}_P{i//MAX_ROWS + 1}"
                        chunk.to_excel(writer, sheet_name=part_name, index=False, freeze_panes=(1,0))
                        num_format = writer.book.add_format({'num_format': '0.00'})
                        writer.sheets[part_name].set_column('K:L', 15, num_format)

    print(f"📁 Report created (handled row limits): {valid_name}")

print(f"\nDone. All reports generated locally for {date_stamp}.")

📁 Report created (handled row limits): India
📁 Report created (handled row limits): WMX Media
📁 Report created (handled row limits): France
📁 Report created (handled row limits): Mexico
📁 Report created (handled row limits): Czech Rep
📁 Report created (handled row limits): Latina
📁 Report created (handled row limits): ATL US
📁 Report created (handled row limits): Vietnam
📁 Report created (handled row limits): Turkey
📁 Report created (handled row limits): UPROXX
📁 Report created (handled row limits): 300 Ent
📁 Report created (handled row limits): Africori
📁 Report created (handled row limits): Elektra
📁 Report created (handled row limits): Philippines
📁 Report created (handled row limits): Nonesuch
📁 Report created (handled row limits): Benelux
📁 Report created (handled row limits): Spain
📁 Report created (handled row limits): Germany
📁 Report created (handled row limits): ATL UK
📁 Report created (handled row limits): Spinnin
📁 Report created (handled row limits): Brazil
📁 Report create